In [56]:
from pathlib import Path

import polars as pl

ROOT = Path.cwd()
OUT_DIR = ROOT / "data" / "output"
if not OUT_DIR.exists():
    OUT_DIR = ROOT.parent / "data" / "output"

parquet_path = OUT_DIR / "dashboard" / "metrics.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [57]:
raw_df.head()

unique_id,wmape,sum_y,n_points,n_with_sales,unidad,n_spine,seccion,store,sku,node_kind
str,f64,f64,u32,u32,str,u32,str,str,str,str
"""23||T:00095||S:578305""",0.747188,345.0,624,2,"""Valor ($)""",624,"""23""","""00095""","""578305""","""tienda_sku"""
"""1||S:411891""",1.0,5.0,0,4,"""Unidades""",null,"""1""",null,"""411891""","""sku"""
"""23||S:455458""",0.755938,232.4,0,4,"""Valor ($)""",null,"""23""",null,"""455458""","""sku"""
"""1||T:00154||S:455436""",0.6875,16.0,731,15,"""Unidades""",731,"""1""","""00154""","""455436""","""tienda_sku"""
"""1||T:00001||S:400744""",0.935549,1950.0,731,6,"""Valor ($)""",731,"""1""","""00001""","""400744""","""tienda_sku"""


In [58]:
best_sku = {
    sec: {
        sto: (
            raw_df.filter(
                (pl.col("node_kind") == "tienda_sku")
                & (pl.col("seccion") == sec)
                & (pl.col("store") == sto)
                & (pl.col("unidad") == "Valor ($)")
                & pl.col("wmape").is_not_null()
                & (pl.col("wmape") != 0)
            )
            .sort("wmape")
            .get_column("sku")
            .head(50)
            .to_list()
        )
        for sto in (
            raw_df.filter(
                (pl.col("node_kind") == "tienda")
                & (pl.col("seccion") == sec)
                & (pl.col("unidad") == "Valor ($)")
            )
            .sort("wmape")
            .get_column("store")
            .head(5)
            .to_list()
        )
    }
    for sec in ["1", "23"]
}

best_sku

{'1': {'00001': ['39472',
   '580661',
   '459977',
   '516028',
   '46499',
   '541427',
   '41796',
   '75048',
   '46853',
   '567332',
   '483046',
   '208',
   '64048',
   '447684',
   '478352',
   '43549',
   '2322',
   '190454',
   '42231',
   '58905',
   '218557',
   '127359',
   '565118',
   '46713',
   '379145',
   '6656',
   '61722',
   '478353',
   '447627',
   '299478',
   '406342',
   '406018',
   '605956',
   '1063',
   '118209',
   '152629',
   '133336',
   '145082',
   '127360',
   '39591',
   '43550',
   '1760',
   '31894',
   '541426',
   '41542',
   '570746',
   '38333',
   '55921',
   '57488',
   '500770'],
  '00003': ['581783',
   '106356',
   '58905',
   '39472',
   '85860',
   '143871',
   '168765',
   '473932',
   '116741',
   '64048',
   '46499',
   '127359',
   '124382',
   '31894',
   '169938',
   '332817',
   '495934',
   '492075',
   '478352',
   '101740',
   '494947',
   '495933',
   '477265',
   '101743',
   '51782',
   '565871',
   '145082',
   '6656',


In [59]:
best_skus = {}
for sec in ["1", "23"]:
    best_skus.update(best_sku[sec])
best_skus

{'00001': ['39472',
  '580661',
  '459977',
  '516028',
  '46499',
  '541427',
  '41796',
  '75048',
  '46853',
  '567332',
  '483046',
  '208',
  '64048',
  '447684',
  '478352',
  '43549',
  '2322',
  '190454',
  '42231',
  '58905',
  '218557',
  '127359',
  '565118',
  '46713',
  '379145',
  '6656',
  '61722',
  '478353',
  '447627',
  '299478',
  '406342',
  '406018',
  '605956',
  '1063',
  '118209',
  '152629',
  '133336',
  '145082',
  '127360',
  '39591',
  '43550',
  '1760',
  '31894',
  '541426',
  '41542',
  '570746',
  '38333',
  '55921',
  '57488',
  '500770'],
 '00003': ['581783',
  '106356',
  '58905',
  '39472',
  '85860',
  '143871',
  '168765',
  '473932',
  '116741',
  '64048',
  '46499',
  '127359',
  '124382',
  '31894',
  '169938',
  '332817',
  '495934',
  '492075',
  '478352',
  '101740',
  '494947',
  '495933',
  '477265',
  '101743',
  '51782',
  '565871',
  '145082',
  '6656',
  '108193',
  '565117',
  '127360',
  '817',
  '40399',
  '555434',
  '494254',
  '

In [60]:
sales_path = OUT_DIR / "sales.parquet"
sales_df = pl.read_parquet(sales_path)
sales_df["STORE_ID", "SKU_ID"].unique()

STORE_ID,SKU_ID
str,str
"""00022""","""365958"""
"""00001""","""545601"""
"""00012""","""559417"""
"""00122""","""109612"""
"""00001""","""228983"""
…,…
"""00001""","""514031"""
"""00003""","""424641"""
"""00063""","""463161"""


In [61]:
best_skus_df = pl.DataFrame(
    [
        {"STORE_ID": sto, "SKU_ID": sku}
        for sto, skus in best_skus.items()
        for sku in skus
    ]
).cast(
    {
        "STORE_ID": pl.String,
        "SKU_ID": pl.String,
    }
)

sales_df.join(
    best_skus_df,
    on=["STORE_ID", "SKU_ID"],
    how="inner",
)

SALES_DATE,SKU_ID,STORE_ID,TRAN_TYPE,SLS_VAL,SLS_QTY,RTRN_QTY,RTRN_VAL
datetime[μs],str,str,str,f64,f64,f64,f64
2025-11-26 00:00:00,"""221258""","""00006""","""P""",3442.66,48.0,0.0,0.0
2025-05-05 00:00:00,"""568161""","""00003""","""R""",1014.43,8.0,0.0,0.0
2026-04-29 00:00:00,"""229229""","""00211""","""R""",64.0,1.0,0.0,0.0
2024-10-04 00:00:00,"""555432""","""00003""","""R""",346.5,8.0,0.0,0.0
2024-12-09 00:00:00,"""532696""","""00095""","""R""",342.0,6.0,0.0,0.0
…,…,…,…,…,…,…,…
2026-04-30 00:00:00,"""568161""","""00211""","""P""",693.0,7.0,0.0,0.0
2025-04-22 00:00:00,"""218058""","""00095""","""P""",953.0,7.0,0.0,0.0
2025-02-12 00:00:00,"""30316""","""00006""","""R""",1008.23,16.0,0.0,0.0


In [62]:
best_sku = {}

sec = "1"

stos = (
    raw_df.filter(
        (pl.col("node_kind") == "tienda")
        & (pl.col("seccion") == sec)
        & (pl.col("unidad") == "Valor ($)")
    )
    .sort("wmape")
    .get_column("store")
    .head(5)
    .to_list()
)

skus = (
    raw_df.filter(
        (pl.col("node_kind") == "tienda_sku")
        & (pl.col("seccion") == sec)
        & (pl.col("unidad") == "Valor ($)")
        & pl.col("store").is_in(stos)
        & pl.col("wmape").is_not_null()
        & (pl.col("wmape") != 0)
        & (pl.col("n_points") == 28)
    )
    .group_by("sku")
    .agg(
        pl.col("store").n_unique().alias("n_stores"),
        pl.col("wmape").mean().alias("wmape_mean"),
    )
    .filter(pl.col("n_stores") == len(stos))
    .sort("wmape_mean")
    .head(50)
    .get_column("sku")
    .to_list()
)

skus

[]

In [63]:
selected_stores = {
    sec: (
        raw_df.filter(
            (pl.col("node_kind") == "tienda")
            & (pl.col("seccion") == sec)
            & (pl.col("unidad") == "Valor ($)")
            & pl.col("wmape").is_not_null()
            & (pl.col("wmape") != 0)
        )
        .sort("wmape")
        .get_column("store")
        .head(5)
        .to_list()
    )
    for sec in ["1", "23"]
}
selected_stores

{'1': ['00001', '00003', '00211', '00154', '00063'],
 '23': ['00006', '00012', '00005', '00046', '00095']}

In [64]:
best_sku = []

for sec, stores in selected_stores.items():
    sku_candidates = (
        raw_df.filter(
            (pl.col("node_kind") == "tienda_sku")
            & (pl.col("seccion") == sec)
            & (pl.col("unidad") == "Valor ($)")
            & pl.col("store").is_in(stores)
            & pl.col("wmape").is_not_null()
            & (pl.col("wmape") != 0)
            & (pl.col("n_points") >= 20)
        )
        .sort("wmape")
        .head(50)
        .get_column("sku")
        .to_list()
    )

    best_sku += sku_candidates

best_sku

['412091',
 '278120',
 '39472',
 '581783',
 '580661',
 '106356',
 '459977',
 '495933',
 '516028',
 '568160',
 '46499',
 '541427',
 '41796',
 '75048',
 '58905',
 '46853',
 '567332',
 '39472',
 '85860',
 '568161',
 '483046',
 '208',
 '143871',
 '489263',
 '606556',
 '64048',
 '447684',
 '568160',
 '478352',
 '43549',
 '565871',
 '2322',
 '168765',
 '190454',
 '473932',
 '42231',
 '58905',
 '456855',
 '218557',
 '568160',
 '43422',
 '127359',
 '116741',
 '100622',
 '565118',
 '46713',
 '379145',
 '6656',
 '495933',
 '61722',
 '603595',
 '195295',
 '219991',
 '587387',
 '113618',
 '514081',
 '582947',
 '333113',
 '113618',
 '501237',
 '113618',
 '218058',
 '103986',
 '71765',
 '448634',
 '113618',
 '32757',
 '45642',
 '71765',
 '197305',
 '474665',
 '28303',
 '333110',
 '197305',
 '306915',
 '593461',
 '306915',
 '218058',
 '5898',
 '392615',
 '598287',
 '600005',
 '582946',
 '45642',
 '385301',
 '519600',
 '448849',
 '17245',
 '126196',
 '25879',
 '419464',
 '523647',
 '288718',
 '466270'

In [65]:
parquet_path = OUT_DIR / "forecast.parquet"
if not parquet_path.exists():
    raise FileNotFoundError(f"Parquet file not found: {parquet_path}")

raw_df = pl.read_parquet(parquet_path)

In [66]:
raw_df

unique_id,ds,value,valuehat,y,yhat,train_start,train_end,test_start,test_end,forecast_start,forecast_end,sku_desc,store_name,seccion,period_type,driver_effect,driver_effect_value,yhat_seccion,valuehat_seccion,yhat_tienda,valuehat_tienda,modelo_seleccionado
str,date,f64,f64,f64,f64,date,date,date,date,date,date,str,str,str,str,f64,f64,f32,f32,f32,f32,str
"""1""",2024-05-26,2.1347e6,2.0427e6,17188.0,16525.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-27,1.9304e6,2.0117e6,16029.0,16911.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-28,2.0906e6,2.2787e6,17250.0,18786.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-29,1.8514e6,1.9099e6,16003.0,16741.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
"""1""",2024-05-30,1.7116e6,1.8177e6,14036.0,15013.0,2024-05-26,2026-03-29,2026-03-30,2026-04-26,2026-04-27,2026-05-26,"""""","""""","""1""","""in_sample""",null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""23||T:00200||S:99008""",2026-01-28,0.0,0.0,0.0,0.0,2024-01-28,2025-11-30,2025-12-01,2025-12-07,2025-12-08,2026-02-01,"""FERNET BRANCA MENTA 750ML""","""ELTANO""","""23""","""forecast_only""",-7.172163,0.081653,21761.0,2.082007e6,1.0,63203.121094,"""tienda"""
"""23||T:00200||S:99008""",2026-01-29,0.0,0.0,0.0,0.0,2024-01-28,2025-11-30,2025-12-01,2025-12-07,2025-12-08,2026-02-01,"""FERNET BRANCA MENTA 750ML""","""ELTANO""","""23""","""forecast_only""",-7.066441,0.184005,24893.0,2416458.5,1.0,70014.789062,"""tienda"""
"""23||T:00200||S:99008""",2026-01-30,0.0,0.0,0.0,0.0,2024-01-28,2025-11-30,2025-12-01,2025-12-07,2025-12-08,2026-02-01,"""FERNET BRANCA MENTA 750ML""","""ELTANO""","""23""","""forecast_only""",-6.7656,0.478391,32150.0,3.2618e6,1.0,93981.40625,"""tienda"""
